# 🚀 Prompt Optimizer — ML Pipeline & Evaluation Report

This notebook demonstrates a complete NLP pipeline for **prompt quality optimization**.  
It uses the OpenRouter API to transform weak prompts into structured, high-quality prompts,  
then evaluates the results with ROUGE, semantic similarity, and custom scoring metrics.

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1 — Install & Import
# ═══════════════════════════════════════════════════════════

import subprocess, sys

# Install all required packages in one go
packages = [
    'python-dotenv',   # Safe .env loading
    'requests',        # HTTP calls to OpenRouter
    'pandas',          # Dataframe manipulation
    'numpy',           # Numeric ops
    'matplotlib',      # Plotting
    'seaborn',         # Statistical plots
    'datasets',        # HuggingFace datasets
    'transformers',    # Tokenizer (flan-t5-base)
    'rouge-score',     # ROUGE metrics
    'sentence-transformers',  # Semantic similarity
    'tqdm',            # Progress bars
    'ipywidgets',      # Interactive demo widget
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

import os, re, time, json, textwrap
import requests as http_requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoTokenizer
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util as st_util
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# Load API key from .env file
load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')
assert API_KEY, '❌ OPENROUTER_API_KEY not found — create a .env file with your key'

# Plot styling for dark backgrounds
plt.style.use('dark_background')
sns.set_palette('viridis')

print('✅ All imports loaded successfully')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2 — Load Dataset
# Two sources combined:
#   A) fka/awesome-chatgpt-prompts (HuggingFace) — auto-degraded
#   B) OpenRouter generation from seed topics
# ═══════════════════════════════════════════════════════════

# ── Source A: HuggingFace awesome-chatgpt-prompts ─────────
# These are high-quality prompts; we degrade them to create weak versions.

def degrade_prompt(good_prompt):
    """Strip structure from a well-crafted prompt to simulate a weak one."""
    text = good_prompt.lower().strip()
    # Remove role assignments
    text = re.sub(r'(act as|you are|as a|as an|i want you to)\s+\w+', '', text)
    # Remove bullet points, numbering, dashes
    text = re.sub(r'[\-\*•]\s+', '', text)
    text = re.sub(r'\d+\.\s+', '', text)
    # Remove formatting cues
    text = re.sub(r'(in bullet points|step by step|in detail|please|provide|explain)', '', text)
    # Collapse whitespace and truncate
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    # Keep only 3–8 words for a believable weak prompt
    return ' '.join(words[:min(8, max(3, len(words) // 4))])

print('Loading HuggingFace dataset...')
hf_data = load_dataset('fka/awesome-chatgpt-prompts', split='train')
source_a = pd.DataFrame({
    'weak_prompt': [degrade_prompt(row['prompt']) for row in hf_data],
    'improved_prompt': [row['prompt'] for row in hf_data],
    'source': 'huggingface'
})
print(f'  Source A: {len(source_a)} rows from HuggingFace')

# ── Source B: OpenRouter-generated from seed topics ───────

SEED_TOPICS = [
    'explain ai', 'python basics', 'climate change', 'healthy eating',
    'machine learning', 'web development', 'photography tips', 'time management',
    'space exploration', 'financial planning', 'creative writing', 'data science',
    'public speaking', 'mental health', 'cybersecurity', 'blockchain',
    'remote work', 'renewable energy', 'artificial intelligence ethics', 'fitness',
    'resume writing', 'social media marketing', 'cloud computing', 'game design',
    'travel planning', 'cooking', 'mindfulness', 'startup ideas',
    'language learning', 'UX design', 'content strategy', 'email writing',
    'project management', 'negotiation skills', 'database design', 'networking',
    'mobile app development', 'video editing', 'SEO optimization', 'podcasting',
]

# System prompt to generate improved versions (same logic as the backend)
SYSTEM_PROMPT = """You are an expert AI prompt engineer. Transform the following weak prompt into a 
highly effective, structured prompt. Include: a clear ROLE, specific ACTION, relevant CONTEXT, 
desired OUTPUT FORMAT, and SPECIFICITY. Return ONLY the improved prompt text."""

# Free-tier model on OpenRouter
OPENROUTER_MODEL = 'mistralai/mistral-small-3.1-24b-instruct:free'

def call_openrouter(prompt, system=SYSTEM_PROMPT, max_tokens=300):
    """Call OpenRouter API and return the generated text."""
    try:
        resp = http_requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {API_KEY}',
                'Content-Type': 'application/json',
                'HTTP-Referer': 'https://prompt-optimizer.vercel.app',
                'X-Title': 'Prompt Optimizer Notebook',
            },
            json={
                'model': OPENROUTER_MODEL,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                'max_tokens': max_tokens,
                'temperature': 0.7,
            },
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content'].strip()
    except Exception as e:
        print(f'  ⚠ API error: {e}')
        return None  # skip failed calls

print('Generating Source B from OpenRouter (this may take a few minutes)...')
source_b_rows = []
for topic in tqdm(SEED_TOPICS, desc='Generating'):
    improved = call_openrouter(topic)
    if improved and len(improved.split()) >= 15:
        source_b_rows.append({
            'weak_prompt': topic,
            'improved_prompt': improved,
            'source': 'openrouter'
        })
    time.sleep(1.0)  # rate-limit courtesy (free tier)

source_b = pd.DataFrame(source_b_rows)
print(f'  Source B: {len(source_b)} rows from OpenRouter')

# ── Combine ──────────────────────────────────────────────
df = pd.concat([source_a, source_b], ignore_index=True)
print(f'\n📊 Combined dataset: {len(df)} total rows')
df.head()

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3 — Data Cleaning & Preprocessing
# ═══════════════════════════════════════════════════════════

print(f'Rows before cleaning: {len(df)}')

# 1. Remove nulls and duplicates
df.dropna(subset=['weak_prompt', 'improved_prompt'], inplace=True)
df.drop_duplicates(subset=['weak_prompt'], inplace=True)

# 2. Strip HTML, excess whitespace, non-ASCII
def clean_text(text):
    text = re.sub(r'<[^>]+>', '', str(text))        # strip HTML tags
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)     # remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()        # collapse whitespace
    return text

df['weak_prompt']     = df['weak_prompt'].apply(lambda x: clean_text(x).lower())
df['improved_prompt'] = df['improved_prompt'].apply(clean_text)

# 3. Filter by length: weak 2–10 words, improved 15–300 words (wider range for robustness)
df['weak_len']     = df['weak_prompt'].apply(lambda x: len(x.split()))
df['improved_len'] = df['improved_prompt'].apply(lambda x: len(x.split()))

df = df[(df['weak_len'] >= 2) & (df['weak_len'] <= 10) &
        (df['improved_len'] >= 15) & (df['improved_len'] <= 300)].copy()

# 4. Add T5-style prefix for the pipeline
df['input_text']  = 'improve prompt: ' + df['weak_prompt']
df['target_text'] = df['improved_prompt']

df.reset_index(drop=True, inplace=True)
print(f'Rows after cleaning: {len(df)}')
print(f'\nWeak prompt length  — min: {df.weak_len.min()}, max: {df.weak_len.max()}, mean: {df.weak_len.mean():.1f}')
print(f'Improved length     — min: {df.improved_len.min()}, max: {df.improved_len.max()}, mean: {df.improved_len.mean():.1f}')

# Show sample rows
df[['weak_prompt', 'improved_prompt', 'source']].sample(min(5, len(df)), random_state=42)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4 — Tokenization
# Uses google/flan-t5-base tokenizer (CPU-only, no training)
# to analyze token distributions and prove NLP pipeline fluency.
# ═══════════════════════════════════════════════════════════

print('Loading Flan-T5 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

# Tokenize inputs and targets
input_encodings  = tokenizer(df['input_text'].tolist(),  padding=True, truncation=True, max_length=128, return_tensors='np')
target_encodings = tokenizer(df['target_text'].tolist(), padding=True, truncation=True, max_length=256, return_tensors='np')

# Compute per-sample token lengths (non-padding tokens)
input_token_lens  = (input_encodings['attention_mask'].sum(axis=1)).tolist()
target_token_lens = (target_encodings['attention_mask'].sum(axis=1)).tolist()

print(f'\nVocabulary size: {tokenizer.vocab_size:,}')
print(f'Input tokens  — mean: {np.mean(input_token_lens):.1f}, max: {np.max(input_token_lens)}')
print(f'Target tokens — mean: {np.mean(target_token_lens):.1f}, max: {np.max(target_token_lens)}')

# Plot token length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(input_token_lens, bins=30, color='#818cf8', edgecolor='#1e1b4b', alpha=0.85)
axes[0].set_title('Input Token Lengths', fontweight='bold')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Count')

axes[1].hist(target_token_lens, bins=30, color='#4ade80', edgecolor='#052e16', alpha=0.85)
axes[1].set_title('Target Token Lengths', fontweight='bold')
axes[1].set_xlabel('Tokens'); axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5 — Train / Validation / Test Split
# 70% train / 15% val / 15% test, stratified by prompt length.
# ═══════════════════════════════════════════════════════════

# Create length bucket for stratification (short / medium / long)
df['length_bucket'] = pd.cut(
    df['weak_len'],
    bins=[0, 3, 6, 100],
    labels=['short', 'medium', 'long']
)

# Check if stratification is possible (need at least 2 per class)
bucket_counts = df['length_bucket'].value_counts()
can_stratify = all(c >= 2 for c in bucket_counts.values)
strat_col = df['length_bucket'] if can_stratify else None

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=strat_col
)

# Second split: 50/50 into val and test (each ~15%)
strat_temp = temp_df['length_bucket'] if can_stratify else None
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=strat_temp
)

print(f'Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}')
print(f'Total: {len(train_df) + len(val_df) + len(test_df)}')

# Distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
split_counts = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Count': [len(train_df), len(val_df), len(test_df)]
})
bars = ax.bar(split_counts['Split'], split_counts['Count'],
              color=['#6366f1', '#a78bfa', '#c4b5fd'], edgecolor='#1e1b4b')
for bar, count in zip(bars, split_counts['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontweight='bold', color='white')
ax.set_title('Train / Validation / Test Split', fontweight='bold')
ax.set_ylabel('Samples')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6 — "Training" via API Integration
# Since we use OpenRouter for inference, "training" =
# running every train sample through the API and storing
# the predictions alongside ground truth for evaluation.
# ═══════════════════════════════════════════════════════════

# Score function (mirrors the backend scorer)
def score_prompt(prompt):
    """Rate a prompt 0–10 on length, role, clarity, format, specificity."""
    text = str(prompt).strip()
    if not text:
        return 0
    words = text.split()
    score = 0

    # Length
    if len(words) >= 20: score += 2
    elif len(words) >= 10: score += 1

    # Role assignment
    role_kw = ['act as', 'you are', 'as a', 'as an', 'role:', 'persona:']
    if any(k in text.lower() for k in role_kw): score += 2

    # Clarity
    actions = ['explain', 'describe', 'list', 'compare', 'write', 'create',
               'summarize', 'analyze', 'generate', 'give me', 'provide', 'show']
    if '?' in text or any(v in text.lower() for v in actions): score += 2
    else: score += 1

    # Format hints
    fmt = ['bullet', 'list', 'step', 'table', 'json', 'markdown',
           'numbered', 'format', 'structure', 'outline', 'paragraph']
    if any(f in text.lower() for f in fmt): score += 2

    # Specificity
    has_nums = bool(re.search(r'\d', text))
    has_proper = bool(re.search(r'[A-Z]', text[1:])) if len(text) > 1 else False
    if has_nums or (has_proper and len(words) >= 15): score += 2
    elif len(words) >= 15: score += 1

    return score

# Run inference on the train split
print(f'Running API inference on {len(train_df)} training samples...')
print('(This will take several minutes due to API rate limits)\n')

api_outputs = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Inference'):
    result = call_openrouter(row['weak_prompt'])
    api_outputs.append(result if result else '')  # empty string on failure
    time.sleep(1.0)  # rate-limit courtesy for free tier

train_df = train_df.copy()
train_df['api_output'] = api_outputs

# Remove rows where API failed
valid_mask = train_df['api_output'].str.len() > 0
print(f'\nSuccessful API calls: {valid_mask.sum()} / {len(train_df)}')
train_df = train_df[valid_mask].copy()

if len(train_df) == 0:
    raise RuntimeError('❌ All API calls failed — check your OPENROUTER_API_KEY in .env')

# Compute scores
train_df['score_original']     = train_df['weak_prompt'].apply(score_prompt)
train_df['score_ground_truth'] = train_df['improved_prompt'].apply(score_prompt)
train_df['score_api_output']   = train_df['api_output'].apply(score_prompt)

# Save intermediate results
train_df.to_csv('results.csv', index=False)
print('\n💾 Results saved to results.csv')
print(f'Mean score gain (API vs original): +{(train_df.score_api_output - train_df.score_original).mean():.2f} points')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7 — Evaluation
# A) ROUGE Score
# B) Semantic Similarity
# C) Prompt Score Comparison
# D) Output Comparison Table
# ═══════════════════════════════════════════════════════════

# ── A. ROUGE Score ────────────────────────────────────────
print('Computing ROUGE scores...')
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for _, row in train_df.iterrows():
    scores = scorer.score(row['improved_prompt'], row['api_output'])
    for key in rouge_results:
        rouge_results[key].append(scores[key].fmeasure)

rouge_means = {k: np.mean(v) if len(v) > 0 else 0.0 for k, v in rouge_results.items()}
print(f"  ROUGE-1: {rouge_means['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_means['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_means['rougeL']:.4f}")

# ROUGE bar chart — with safe ylim to avoid NaN/Inf crash
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rouge_means.keys(), rouge_means.values(),
              color=['#6366f1', '#8b5cf6', '#a78bfa'], edgecolor='#1e1b4b')
for bar, val in zip(bars, rouge_means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold', color='white', fontsize=11)
ax.set_title('ROUGE Scores (API Output vs Ground Truth)', fontweight='bold')
max_rouge = max(rouge_means.values()) if rouge_means.values() else 0
ax.set_ylim(0, max(max_rouge * 1.25, 0.1))  # ensure valid, non-zero upper limit
plt.tight_layout(); plt.show()

# ── B. Semantic Similarity ────────────────────────────────
print('\nComputing semantic similarity...')
sem_model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight, CPU-friendly

gt_embeddings  = sem_model.encode(train_df['improved_prompt'].tolist(), show_progress_bar=True)
api_embeddings = sem_model.encode(train_df['api_output'].tolist(), show_progress_bar=True)

# Pairwise cosine similarity
cos_sims = [float(st_util.cos_sim(gt_embeddings[i], api_embeddings[i])) for i in range(len(gt_embeddings))]
train_df['semantic_similarity'] = cos_sims

mean_sim = np.mean(cos_sims) if cos_sims else 0.0
print(f'  Mean semantic similarity: {mean_sim:.4f}')

# Histogram of similarity scores
fig, ax = plt.subplots(figsize=(8, 4))
if cos_sims:
    ax.hist(cos_sims, bins=30, color='#4ade80', edgecolor='#052e16', alpha=0.85)
    ax.axvline(mean_sim, color='#f59e0b', linestyle='--', linewidth=2, label=f'Mean: {mean_sim:.4f}')
ax.set_title('Semantic Similarity Distribution', fontweight='bold')
ax.set_xlabel('Cosine Similarity'); ax.set_ylabel('Count')
ax.legend(); plt.tight_layout(); plt.show()

# ── C. Prompt Score Comparison (grouped bar chart) ────────
print('\nComputing prompt score comparison...')
score_means = {
    'Original':     train_df['score_original'].mean(),
    'Ground Truth': train_df['score_ground_truth'].mean(),
    'API Output':   train_df['score_api_output'].mean(),
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#ef4444', '#22c55e', '#6366f1']
bars = ax.bar(score_means.keys(), score_means.values(), color=colors, edgecolor='#111827')
for bar, val in zip(bars, score_means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontweight='bold', color='white', fontsize=12)
ax.set_title('Prompt Quality Scores: Original vs Ground Truth vs API Output', fontweight='bold')
ax.set_ylabel('Score (0–10)'); ax.set_ylim(0, 11)
plt.tight_layout(); plt.show()

# ── D. Output Comparison Table ────────────────────────────
print('\nSide-by-side comparison (10 samples):')
comparison = train_df[['weak_prompt', 'improved_prompt', 'api_output',
                        'score_original', 'score_api_output']].head(10).copy()
comparison.columns = ['Weak Prompt', 'Ground Truth', 'API Output', 'Score Before', 'Score After']
# Truncate long text for display
for col in ['Ground Truth', 'API Output']:
    comparison[col] = comparison[col].apply(lambda x: textwrap.shorten(str(x), width=80, placeholder='…'))
comparison

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8 — Report Generation
# Prints a formatted summary and exports report.csv
# ═══════════════════════════════════════════════════════════

avg_gain = (train_df['score_api_output'] - train_df['score_original']).mean()
total_samples = len(df)

# Identify best/worst performing topics by score gain
train_df['score_gain'] = train_df['score_api_output'] - train_df['score_original']
best_topic  = train_df.loc[train_df['score_gain'].idxmax(), 'weak_prompt']
worst_topic = train_df.loc[train_df['score_gain'].idxmin(), 'weak_prompt']

report = f"""
═══════════════════════════════════════
       PROMPT OPTIMIZER — RESULTS
═══════════════════════════════════════
Dataset        : {total_samples} samples
Train/Val/Test : {len(train_df)} / {len(val_df)} / {len(test_df)}

ROUGE-1        : {rouge_means['rouge1']:.4f}
ROUGE-2        : {rouge_means['rouge2']:.4f}
ROUGE-L        : {rouge_means['rougeL']:.4f}
Semantic Sim   : {mean_sim:.4f}
Avg Score Gain : +{avg_gain:.1f} points (out of 10)

Best performing topic  : {best_topic}
Worst performing topic : {worst_topic}
═══════════════════════════════════════
"""
print(report)

# Export summary metrics to CSV
report_data = pd.DataFrame({
    'Metric': ['Dataset Size', 'Train', 'Validation', 'Test',
               'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Semantic Similarity',
               'Avg Score Gain', 'Best Topic', 'Worst Topic'],
    'Value': [total_samples, len(train_df), len(val_df), len(test_df),
              f"{rouge_means['rouge1']:.4f}", f"{rouge_means['rouge2']:.4f}",
              f"{rouge_means['rougeL']:.4f}", f'{mean_sim:.4f}',
              f'+{avg_gain:.1f}', best_topic, worst_topic]
})
report_data.to_csv('report.csv', index=False)
print('💾 Report exported to report.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9 — Live Demo (interactive widget)
# Tests the end-to-end pipeline inside the notebook itself.
# ═══════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, HTML

# Input widget
prompt_input = widgets.Textarea(
    value='',
    placeholder='Type a weak prompt, e.g. "explain ai"',
    description='Prompt:',
    layout=widgets.Layout(width='90%', height='80px')
)

# Output area
output_area = widgets.Output()

# Optimize button callback
def on_optimize(btn):
    output_area.clear_output()
    weak = prompt_input.value.strip()
    if not weak:
        with output_area:
            print('⚠️  Please enter a prompt first.')
        return

    with output_area:
        print('⏳ Optimizing via OpenRouter...')
        improved = call_openrouter(weak)
        if not improved:
            print('❌ API call failed. Check your API key and try again.')
            return

        before = score_prompt(weak)
        after  = score_prompt(improved)

        display(HTML(f"""
        <div style="background:#1e1b4b; padding:16px; border-radius:12px; margin-top:8px; font-family:monospace;">
            <div style="color:#a5b4fc; font-weight:bold; margin-bottom:8px;">🔹 Original (Score: {before}/10)</div>
            <div style="color:#9ca3af; margin-bottom:16px;">{weak}</div>
            <div style="color:#4ade80; font-weight:bold; margin-bottom:8px;">🚀 Optimized (Score: {after}/10)</div>
            <div style="color:#e5e7eb;">{improved}</div>
            <div style="margin-top:12px; color:#f59e0b; font-weight:bold;">Score gain: +{after - before} points</div>
        </div>
        """))

optimize_btn = widgets.Button(
    description='🚀 Optimize',
    button_style='info',
    layout=widgets.Layout(width='200px', height='36px')
)
optimize_btn.on_click(on_optimize)

# Display the interactive demo
display(HTML('<h3 style="color:#818cf8;">🎯 Live Demo — Try it yourself</h3>'))
display(prompt_input)
display(optimize_btn)
display(output_area)